# IHGAMP - Step 3: Model Training

Train HRD prediction model:
- StandardScaler → PCA → Ridge Regression → Platt Calibration
- 5-fold stratified cross-validation

In [ ]:
import sys
sys.path.append('..')

from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

from src.models import HRDPredictor

OUTPUT_DIR = Path('./outputs')
SEED = 42
np.random.seed(SEED)

In [ ]:
# Load data
embeddings = pd.read_parquet(OUTPUT_DIR / 'embeddings' / 'patient_embeddings.parquet')
labels = pd.read_csv('data/hrd_labels.csv')  # Expected: patient, HRD

df = embeddings.merge(labels, on='patient', how='inner')
print(f'Matched: {len(df)} patients')

In [ ]:
# Binarize HRD (top 20% = HRD-high)
threshold = np.percentile(df['HRD'], 80)
df['label'] = (df['HRD'] >= threshold).astype(int)
print(f'HRD threshold: {threshold:.2f}')
print(f'Class distribution: {df["label"].value_counts().to_dict()}')

In [ ]:
# Train/val/test split
train_df, temp_df = train_test_split(df, test_size=0.3, stratify=df['label'], random_state=SEED)
val_df, test_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df['label'], random_state=SEED)

df.loc[train_df.index, 'split'] = 'train'
df.loc[val_df.index, 'split'] = 'val'
df.loc[test_df.index, 'split'] = 'test'

z_cols = [c for c in df.columns if c.startswith('z')]
X_train = df.loc[df['split'] == 'train', z_cols].values
y_train = df.loc[df['split'] == 'train', 'HRD'].values
print(f'Train: {len(X_train)}')

In [ ]:
# Train model
model = HRDPredictor(pca_components=128, ridge_alpha=1.0, calibration='platt', seed=SEED)
model.fit(X_train, y_train)
print('Model trained')

# Cross-validation predictions
oof_train = model.cross_val_predict(X_train, y_train, n_splits=5)

In [ ]:
# Generate all predictions
X_val = df.loc[df['split'] == 'val', z_cols].values
X_test = df.loc[df['split'] == 'test', z_cols].values

df['pred'] = np.nan
df.loc[df['split'] == 'train', 'pred'] = oof_train
df.loc[df['split'] == 'val', 'pred'] = model.predict_proba(X_val)
df.loc[df['split'] == 'test', 'pred'] = model.predict_proba(X_test)

In [ ]:
# Save
MODELS_DIR = OUTPUT_DIR / 'models'
MODELS_DIR.mkdir(parents=True, exist_ok=True)
model.save(str(MODELS_DIR / 'hrd_predictor.joblib'))

df[['patient', 'split', 'HRD', 'label', 'pred']].to_csv(OUTPUT_DIR / 'predictions.csv', index=False)
print('Model and predictions saved')